# Week 5 Day 1 — Agent Foundations
## Reasoning Loops, Tool Calling & Raw Python Agents
Today we will build a minimal AI agent from scratch using Python and the Anthropic API.
We will NOT use:
- LangChain
- LangGraph
- CrewAI
- AutoGen
The purpose is to understand what an agent actually does underneath these frameworks.

## API Provider Note

The original task required the **Anthropic API**, but its paid API setup was not available for this exercise.
I initially tried the **Gemini API**, but access was denied because the Google Cloud project had no billing account.
I therefore switched to **Groq** using the OpenAI-compatible API with the `llama-3.3-70b-versatile` model.
The same agent architecture was implemented using Groq, including tool calling and reasoning loops.
This allowed me to continue learning **tool use, memory, state handling, and raw Python agents** without agent frameworks.


# Task 1 — Agent Concepts & Mental Model
## Chatbot vs Workflow vs Agent
### Chatbot
A chatbot mainly receives a user message and generates a response.
```text
User → LLM → Response

### Workflow
A workflow follows a predefined sequence of steps.
Input
  ↓
Step 1
  ↓
Step 2
  ↓
Step 3
  ↓
Output
The developer decides the execution path.

### Agent
An agent gives the LLM some responsibility for deciding what to do next.
User Task
    ↓
   LLM
    ↓
Choose Action
    ↓
Use Tool
    ↓
Observe Result
    ↓
   LLM
    ↓
Another Action OR Final Answer.
The key difference is that the agent can dynamically choose its next action based on the current task and observations.

#  What Makes Something Agentic?
### Markdown cell

## What Makes a System Agentic?
A system becomes more agentic when it has several of the following capabilities:
### 1. Autonomy
The system can decide what action to take next instead of following only a completely fixed sequence.
### 2. Tool Use
The model can interact with external capabilities such as:

- Calculator
- Weather API
- Database
- Web search
- File system
- Python execution
### 3. Multi-Step Planning
The agent can break a problem into several actions.
For example:
Compare weather in Lahore and Islamabad

1. Get Lahore weather
2. Get Islamabad weather
3. Compare temperatures
4. Generate answer

4. Observation
After performing an action, the agent receives the result.

5. Self-Correction
If a tool fails, the agent can potentially decide what to do next.

###### Agentic behavior is not simply "using an LLM." It is an LLM participating in a controlled decision-making loop.

# ReAct
### ReAct Pattern
ReAct means:
**Reason → Act → Observe → Repeat**
The basic architecture is:

             USER TASK
                 ↓
              REASON
                 ↓
                ACT
                 ↓
              OBSERVE
                 ↓
              REASON
                 ↓
                ACT
                 ↓
              OBSERVE
                 ↓
              FINISH

User:

"Which city is warmer, Lahore or Islamabad?"

The agent may decide:

Reason:
I need Lahore's temperature.

Act:
Call weather(Lahore).

Observe:
Lahore = 38°C.

Reason:
I still need Islamabad's temperature.

Act:
Call weather(Islamabad).

Observe:
Islamabad = 31°C.

Reason:
38 > 31.

Final:
Lahore is warmer by 7°C.

##### pseudocode
while not finished:
    response = llm(task)
    if response.requests_tool:
        result = execute_tool(
            response.tool_name,
            response.arguments
        )
        send_result_back_to_llm(result)
    else:
        return response

#### When Is an Agent Overkill?
An agent is unnecessary when the problem has a simple and deterministic solution.
For example, if I only need to calculate:
25 × 40

I do not need an autonomous agent.

Similarly, if a data pipeline always follows:

Read CSV
    ↓
Clean Data
    ↓
Train Model
    ↓
Evaluate Model

a normal Python workflow is usually better.

# Task 2 — Tool Calling Fundamentals
Tool calling allows the Gemini model to request an external function when it needs information or computation that it cannot reliably perform by itself.

In this notebook, the model has access to two tools:

1. `calculator` — performs exact arithmetic.
2. `get_weather` — retrieves weather information from our local weather dataset.

The important architectural idea is that Gemini does not directly execute our Python functions. Gemini produces a structured function call, our Python application executes the corresponding function, and the result is returned to Gemini as a tool response.

In [53]:
import os
import json
import ast
import operator
from openai import OpenAI
from dotenv import load_dotenv
from google import genai

load_dotenv()

print("Libraries imported successfully.")

Libraries imported successfully.


In [54]:
import os
print(bool(os.getenv("XAI_API_KEY")))

True


In [55]:
from google.genai import types
print("Gemini tool-calling types imported successfully.")

Gemini tool-calling types imported successfully.


In [59]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

api_key = os.getenv("XAI_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "llama-3.3-70b-versatile"

print("Groq client created successfully.")
print("Model:", MODEL)

Groq client created successfully.
Model: llama-3.3-70b-versatile


In [62]:
response = client.responses.create(
    model=MODEL,
    input="Say hello in one sentence."
)

print(response.output_text)

Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [63]:
#calculator
_ALLOWED_BINOPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
}
_ALLOWED_UNARYOPS = {
    ast.UAdd: operator.pos,
    ast.USub: operator.neg,
}
def _evaluate_math_node(node):

    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value

    if isinstance(node, ast.BinOp):

        if type(node.op) not in _ALLOWED_BINOPS:
            raise ValueError("Unsupported operator.")

        left = _evaluate_math_node(node.left)
        right = _evaluate_math_node(node.right)

        return _ALLOWED_BINOPS[type(node.op)](
            left,
            right
        )

    if isinstance(node, ast.UnaryOp):

        if type(node.op) not in _ALLOWED_UNARYOPS:
            raise ValueError("Unsupported unary operator.")

        return _ALLOWED_UNARYOPS[type(node.op)](
            _evaluate_math_node(node.operand)
        )

    raise ValueError("Unsupported expression.")

def calculator(expression):
    try:
        tree = ast.parse(
            expression,
            mode="eval"
        )
        result = _evaluate_math_node(tree.body)
        return {
            "result": result
        }
    except Exception as e:

        return {
            "error": str(e)
        }

In [64]:
print(calculator("25 * 40"))
print(calculator("100 / 4"))
print(calculator("2 ** 8"))

{'result': 1000}
{'result': 25.0}
{'result': 256}


In [65]:
#weather
WEATHER_DATA = {
    "Lahore": {
        "temperature_c": 38,
        "condition": "Sunny"
    },
    "Islamabad": {
        "temperature_c": 31,
        "condition": "Partly cloudy"
    },
    "Karachi": {
        "temperature_c": 34,
        "condition": "Humid"
    }
}

In [66]:
def get_weather(city):
    city = city.strip()
    if city not in WEATHER_DATA:
        return {
            "error": f"Weather data unavailable for {city}."
        }
    return {
        "city": city,
        **WEATHER_DATA[city]
    }

In [67]:
print(get_weather("Lahore"))
print(get_weather("Faisalabad"))
print(get_weather("Islamabad"))

{'city': 'Lahore', 'temperature_c': 38, 'condition': 'Sunny'}
{'error': 'Weather data unavailable for Faisalabad.'}
{'city': 'Islamabad', 'temperature_c': 31, 'condition': 'Partly cloudy'}


In [106]:
calculator_tool = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": (
            "Perform exact arithmetic calculations. "
            "Use this tool whenever a numerical calculation is required."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A mathematical expression such as '25 * 40'."
                }
            },
            "required": ["expression"]
        }
    }
}


weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": (
            "Get weather information for a city. "
            "Use this tool when the user asks about weather or temperature."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The city whose weather is requested."
                }
            },
            "required": ["city"]
        }
    }
}


tools = [
    calculator_tool,
    weather_tool
]

print(json.dumps(tools, indent=2))

[
  {
    "type": "function",
    "function": {
      "name": "calculator",
      "description": "Perform exact arithmetic calculations. Use this tool whenever a numerical calculation is required.",
      "parameters": {
        "type": "object",
        "properties": {
          "expression": {
            "type": "string",
            "description": "A mathematical expression such as '25 * 40'."
          }
        },
        "required": [
          "expression"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Get weather information for a city. Use this tool when the user asks about weather or temperature.",
      "parameters": {
        "type": "object",
        "properties": {
          "city": {
            "type": "string",
            "description": "The city whose weather is requested."
          }
        },
        "required": [
          "city"
        ]
      }
    }
  }
]


In [74]:
user_query = "What is 25 * 40?"

response = client.responses.create(
    model=MODEL,
    input=user_query,
    tools=tools
)

print("Grok response received.")

for item in response.output:
    print(item)

Grok response received.
ResponseReasoningItem(id='resp_01kznkb194esjtrg1hh5zffcv1', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed')
ResponseFunctionToolCall(arguments='{"expression":"25 * 40"}', call_id='dq3heqxfa', name='calculator', type='function_call', id='dq3heqxfa', caller=None, namespace=None, status='completed')


In [76]:
print("Response type:", type(response))

for item in response.output:
    print("Item:", item)
    print("Type:", item.type)

    if item.type == "function_call":
        print("\nFunction name:")
        print(item.name)

        print("\nArguments:")
        print(item.arguments)

        print("\nCall ID:")
        print(item.call_id)

Response type: <class 'openai.types.responses.response.Response'>
Item: ResponseReasoningItem(id='resp_01kznkb194esjtrg1hh5zffcv1', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed')
Type: reasoning
Item: ResponseFunctionToolCall(arguments='{"expression":"25 * 40"}', call_id='dq3heqxfa', name='calculator', type='function_call', id='dq3heqxfa', caller=None, namespace=None, status='completed')
Type: function_call

Function name:
calculator

Arguments:
{"expression":"25 * 40"}

Call ID:
dq3heqxfa


In [78]:
import json

tool_call = None

for item in response.output:
    if item.type == "function_call":
        tool_call = item
        break

if tool_call is None:
    print("No tool call was returned.")
else:
    print("Tool:", tool_call.name)
    print("Arguments:", json.loads(tool_call.arguments))
    print("Call ID:", tool_call.call_id)

Tool: calculator
Arguments: {'expression': '25 * 40'}
Call ID: dq3heqxfa


In [80]:
tool_call = None

for item in response.output:

    if item.type == "function_call":
        tool_call = item
        break

if tool_call:
    print("Tool:", tool_call.name)
    print("Arguments:", tool_call.arguments)
    print("Call ID:", tool_call.call_id)
else:
    print("No tool call.")

Tool: calculator
Arguments: {"expression":"25 * 40"}
Call ID: dq3heqxfa


In [84]:
tool_name = tool_call.name
tool_args = json.loads(tool_call.arguments)

if tool_name not in available_tools:
    tool_result = {"error": f"Unknown tool: {tool_name}"}
else:
    tool_result = available_tools[tool_name](**tool_args)

print("Tool result:", tool_result)

# Build the full input history: original user turn + the model's function_call
# item + our function_call_output, all sent together (no server-side chaining)
input_items = [
    {"role": "user", "content": user_query},
    *response.output,
    {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(tool_result)
    }
]

final_response = client.responses.create(
    model=MODEL,
    input=input_items,
    tools=tools
)

print("\nFinal Answer:")
print(final_response.output_text)

Tool result: {'result': 1000}

Final Answer:
The result of 25 * 40 is 1000.


In [83]:
available_tools = {
    "calculator": calculator,
    "get_weather": get_weather
}

print("Available tools:")
for name in available_tools:
    print("-", name)

Available tools:
- calculator
- get_weather


In [88]:
user_query = "What is the weather in Lahore?"

response = client.responses.create(
    model=MODEL,
    input=user_query,
    tools=tools
)

print("Groq response received.")

tool_call = None
for item in response.output:
    if item.type == "function_call":
        tool_call = item
        break

if tool_call:
    print("Tool:", tool_call.name)
    print("Arguments:", tool_call.arguments)
    print("Call ID:", tool_call.call_id)
else:
    print("No tool call returned.")

Groq response received.
Tool: get_weather
Arguments: {"city":"Lahore"}
Call ID: ys9mw8c6j


In [86]:
tool_name = tool_call.name
tool_args = json.loads(tool_call.arguments)

if tool_name not in available_tools:

    tool_result = {
        "error": f"Unknown tool: {tool_name}"
    }

else:

    tool_result = available_tools[tool_name](**tool_args)

print("Tool name:", tool_name)
print("Tool arguments:", tool_args)
print("Tool result:", tool_result)

Tool name: calculator
Tool arguments: {'expression': '25 * 40'}
Tool result: {'result': 1000}


In [68]:
tool_response_part = types.Part.from_function_response(
    name=tool_name,
    response=tool_result
)

follow_up_contents = [
    user_query,
    response.candidates[0].content,
    types.Content(
        role="user",
        parts=[tool_response_part]
    )
]

final_response = client.models.generate_content(
    model=MODEL,
    contents=follow_up_contents,
    config=types.GenerateContentConfig(
        tools=[tools]
    )
)

print("Final answer:")
print(final_response.text)

Final answer:
The current weather in Lahore is sunny with a temperature of 38°C.


In [89]:
input_items = [
    {"role": "user", "content": user_query},
    *response.output,
    {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(tool_result)
    }
]

final_response = client.responses.create(
    model=MODEL,
    input=input_items,
    tools=tools
)

print("Final answer:")
print(final_response.output_text)

Final answer:
The weather in Lahore is partly cloudy with a high temperature of 28 degrees Celsius and a low of 18 degrees Celsius.


#### Task 2 — Key Observations
The calculator and weather tools were both tested through the complete Gemini function-calling cycle.

For the calculator example, Gemini selected the `calculator` tool and generated the argument `expression="25 * 40"`. The Python application then executed the calculator function and returned `{"result": 1000}` to Gemini, which produced the final answer.

For the weather example, Gemini selected the `get_weather` tool and generated the argument `city="Lahore"`. The Python application executed the weather function and returned the weather observation to Gemini, which then generated a natural-language response.

The `available_tools` dictionary acts as a tool registry. Instead of manually writing an `if/elif` statement for every possible tool, the agent dynamically maps the model's requested tool name to the corresponding Python function using:

`available_tools[tool_name](**tool_args)`

This approach is easier to maintain and scale because new tools can be added to the registry without changing the dispatching logic.

Tool descriptions were also important because they provide the semantic information Gemini uses when deciding which tool to call. Clear descriptions reduce incorrect tool selection and missed tool calls by explaining what each tool does, when it should be used, and what its parameters represent.

## Task 3 — Build a Minimal Agent Loop

In Task 2, we manually handled one tool call from Gemini.

In this task, we will turn that process into an actual agent loop.

The agent will repeatedly:

1. Send the current conversation state to Gemini.
2. Check whether Gemini requested a tool.
3. Execute the requested Python tool.
4. Return the tool result to Gemini.
5. Repeat the process if another tool is required.
6. Stop when Gemini produces a final text response.

A maximum iteration limit will be added as a safety mechanism so that an unexpected model behavior cannot cause the program to run forever.

### Agent Loop Architecture

The minimal agent follows this cycle:

User Task
    ↓
Gemini
    ↓
Does Gemini request a tool?
    │
    ├── No → Final Answer → STOP
    │
    └── Yes
          ↓
       Execute Tool
          ↓
       Tool Result
          ↓
       Add Observation
          ↓
       Gemini
          ↓
       Repeat

This is the core idea behind a ReAct-style agent:

Reason → Act → Observe → Repeat

In [115]:
def create_working_memory(user_query):
    return {
        "user_query": user_query,
        "iteration": 0,
        "tool_calls": [],
        "observations": [],
        "errors": []
    }


def log_event(event_type, message):
    print(f"[{event_type}] {message}")


def run_agent(user_query, max_iterations=5):

    print("=" * 70)
    print("AGENT STARTED")
    print("=" * 70)

    # ==================================================
    # 1. CONVERSATION MEMORY
    # ==================================================

    messages = [
        {
            "role": "user",
            "content": user_query
        }
    ]

    # ==================================================
    # 2. WORKING MEMORY
    # ==================================================

    state = create_working_memory(user_query)

    log_event("USER", user_query)

    # ==================================================
    # 3. AGENT LOOP
    # ==================================================

    iteration = 1

    while iteration <= max_iterations:

        state["iteration"] = iteration

        log_event(
            "REASON",
            f"Model is deciding what to do. Iteration {iteration}"
        )

        # ==================================================
        # ASK MODEL
        # ==================================================

        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=tools,
                tool_choice="auto"
            )

            message = response.choices[0].message

        except Exception as e:

            log_event("ERROR", f"Model call failed: {e}")

            state["errors"].append({
                "error": "Model call failed (likely malformed tool-call generation).",
                "details": str(e)
            })

            final_answer = (
                "I'm sorry, I wasn't able to process that request due to an "
                "internal generation error. Could you rephrase or provide more detail?"
            )

            print("\nWorking Memory:")
            print(json.dumps(state, indent=2, default=str))

            return final_answer, state

        # ==================================================
        # NO TOOL CALL → FINAL ANSWER
        # ==================================================

        if not message.tool_calls:

            final_answer = message.content

            log_event("FINAL", "Model did not request another tool.")

            print("\nFinal Answer:")
            print(final_answer)

            print("\nWorking Memory:")
            print(json.dumps(state, indent=2, default=str))

            print("\n" + "=" * 70)
            print("AGENT FINISHED")
            print("=" * 70)

            return final_answer, state

        assistant_message = {
            "role": "assistant",
            "content": message.content,
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments
                    }
                }
                for tc in message.tool_calls
            ]
        }

        messages.append(assistant_message)

        # ==================================================
        # EXECUTE TOOL CALLS
        # ==================================================

        for tool_call in message.tool_calls:

            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            # ----------------------------------------
            # ACT
            # ----------------------------------------

            log_event("ACT", f"Calling tool '{tool_name}' with {tool_args}")

            state["tool_calls"].append({
                "name": tool_name,
                "arguments": tool_args
            })

            # ----------------------------------------
            # TOOL LOOKUP
            # ----------------------------------------

            if tool_name not in available_tools:

                tool_result = {"error": f"Unknown tool: {tool_name}"}
                state["errors"].append(tool_result)
                log_event("ERROR", str(tool_result))

            else:

                try:
                    tool_result = available_tools[tool_name](**tool_args)

                except Exception as e:
                    tool_result = {"error": str(e)}
                    state["errors"].append(tool_result)
                    log_event("ERROR", str(tool_result))

            # ----------------------------------------
            # OBSERVE
            # ----------------------------------------

            log_event("OBSERVE", f"Tool '{tool_name}' returned {tool_result}")

            state["observations"].append({
                "tool": tool_name,
                "result": tool_result
            })

            # ----------------------------------------
            # SEND TOOL RESULT BACK TO MODEL
            # ----------------------------------------

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_name,
                "content": json.dumps(tool_result)
            })

        # ==================================================
        # STATE UPDATED
        # ==================================================

        log_event("STATE", "Conversation memory updated with tool results.")

        iteration += 1

    # ==================================================
    # MAX ITERATIONS REACHED
    # ==================================================

    log_event("ERROR", "Maximum iterations reached.")

    state["errors"].append({"error": "Maximum iterations reached."})

    print("\nWorking Memory:")
    print(json.dumps(state, indent=2, default=str))

    return {"error": "Maximum iterations reached."}, state

In [100]:
result = run_agent(
    "What is 125 * 8?"
)
print("\nReturned result:")
print(result)

AGENT STARTED
[USER] What is 125 * 8?
[REASON] Model is deciding what to do. Iteration 1
[ACT] Calling tool 'calculator' with {'expression': '125 * 8'}
[OBSERVE] Tool 'calculator' returned {'result': 1000}
[STATE] Conversation memory updated with model response and tool results.
[REASON] Model is deciding what to do. Iteration 2
[FINAL] Model did not request another tool.

Final Answer:
The answer is 1000.

Working Memory:
{
  "user_query": "What is 125 * 8?",
  "iteration": 2,
  "tool_calls": [
    {
      "name": "calculator",
      "arguments": {
        "expression": "125 * 8"
      }
    }
  ],
  "observations": [
    {
      "tool": "calculator",
      "result": {
        "result": 1000
      }
    }
  ],
  "errors": []
}

AGENT FINISHED

Returned result:
('The answer is 1000.', {'user_query': 'What is 125 * 8?', 'iteration': 2, 'tool_calls': [{'name': 'calculator', 'arguments': {'expression': '125 * 8'}}], 'observations': [{'tool': 'calculator', 'result': {'result': 1000}}], 'er

In [81]:
result = run_agent(
    "What is the weather in Lahore?"
)
print("\nReturned result:")
print(result)

AGENT STARTED
User: What is the weather in Lahore?

--- Iteration 1 ---

Tool requested: get_weather
Tool arguments: {'city': 'Lahore'}
Tool result: {'city': 'Lahore', 'temperature_c': 38, 'condition': 'Sunny'}

--- Iteration 2 ---
No tool call requested.

Final Answer:
The current weather in Lahore is sunny with a temperature of 38°C.

AGENT FINISHED

Returned result:
The current weather in Lahore is sunny with a temperature of 38°C.


In [82]:
result = run_agent(
    "Look up the weather in Lahore and Islamabad and tell me which city is warmer."
)
print("\nReturned result:")
print(result)

AGENT STARTED
User: Look up the weather in Lahore and Islamabad and tell me which city is warmer.

--- Iteration 1 ---

Tool requested: get_weather
Tool arguments: {'city': 'Lahore'}
Tool result: {'city': 'Lahore', 'temperature_c': 38, 'condition': 'Sunny'}

Tool requested: get_weather
Tool arguments: {'city': 'Islamabad'}
Tool result: {'city': 'Islamabad', 'temperature_c': 31, 'condition': 'Partly cloudy'}

--- Iteration 2 ---
No tool call requested.

Final Answer:
The weather in both cities is as follows:

* **Lahore:** 38°C (Sunny)
* **Islamabad:** 31°C (Partly cloudy)

**Lahore** is warmer than Islamabad by 7°C.

AGENT FINISHED

Returned result:
The weather in both cities is as follows:

* **Lahore:** 38°C (Sunny)
* **Islamabad:** 31°C (Partly cloudy)

**Lahore** is warmer than Islamabad by 7°C.


In [84]:
result = run_agent(
    "Look up the weather in Lahore and Islamabad and tell me which city is warmer.",
    max_iterations=1
)
print("\nReturned result:")
print(result)

AGENT STARTED
User: Look up the weather in Lahore and Islamabad and tell me which city is warmer.

--- Iteration 1 ---

Tool requested: get_weather
Tool arguments: {'city': 'Lahore'}
Tool result: {'city': 'Lahore', 'temperature_c': 38, 'condition': 'Sunny'}

Tool requested: get_weather
Tool arguments: {'city': 'Islamabad'}
Tool result: {'city': 'Islamabad', 'temperature_c': 31, 'condition': 'Partly cloudy'}

Maximum iterations reached.
Agent stopped to prevent an infinite loop.

Returned result:
{'error': 'Maximum iterations reached.'}


## Task 3 — What We Built
The previous single-tool demonstration has now been converted into a reusable agent loop.

The `run_agent()` function sends the current state to Gemini and checks whether the model requested one or more tools. When a tool call is returned, the agent extracts the tool name and arguments, looks up the corresponding Python function from the `available_tools` registry, executes it, and sends the resulting observation back to Gemini.

The loop continues until Gemini no longer requests a tool and instead produces a final text response.

The multi-step weather test demonstrates why the loop is necessary. The agent needs information about both Lahore and Islamabad before it can compare their temperatures. It therefore performs multiple tool interactions before producing the final answer.

The `max_iterations` parameter provides an important safety mechanism. Without it, unexpected model behavior could potentially cause the agent to continue requesting tools indefinitely. In production systems, iteration limits, timeouts, token limits, and other guardrails are commonly used to control agent execution.

# Task 4 — Memory & State Handling
An agent needs state because its task may require multiple interactions with the model and external tools.

There are two important forms of memory in our minimal agent:

1. Conversation memory
2. Working memory

Conversation memory contains the messages and tool results that are passed back to the model so that Gemini can understand what has already happened.

Working memory is the structured state maintained by the application while the agent is executing a task. It can contain information such as the current iteration, tools used, observations collected, errors, and other intermediate information useful for controlling or debugging the agent.

In this task, we will add both forms of state to our raw Python agent.

## Memory Architecture

                    AGENT
                      │
          ┌───────────┴───────────┐
          │                       │
          ▼                       ▼
 Conversation Memory       Working Memory
          │                       │
          │                       ├── iteration
          │                       ├── tools used
          │                       ├── observations
          │                       └── errors
          │
          ├── user message
          ├── model response
          ├── tool call
          └── tool result
          │
          ▼
       Gemini

In [102]:
state = create_working_memory(
    "Compare the weather in Lahore and Islamabad."
)
print(state)

{'user_query': 'Compare the weather in Lahore and Islamabad.', 'iteration': 0, 'tool_calls': [], 'observations': [], 'errors': []}


In [107]:
result, state = run_agent(
    "Look up the weather in Lahore and Islamabad and tell me which city is warmer."
)
print("\nReturned Answer:")
print(result)

AGENT STARTED
[USER] Look up the weather in Lahore and Islamabad and tell me which city is warmer.
[REASON] Model is deciding what to do. Iteration 1
[ACT] Calling tool 'get_weather' with {'city': 'Lahore'}
[OBSERVE] Tool 'get_weather' returned {'city': 'Lahore', 'temperature_c': 38, 'condition': 'Sunny'}
[ACT] Calling tool 'get_weather' with {'city': 'Islamabad'}
[OBSERVE] Tool 'get_weather' returned {'city': 'Islamabad', 'temperature_c': 31, 'condition': 'Partly cloudy'}
[STATE] Conversation memory updated with tool results.
[REASON] Model is deciding what to do. Iteration 2
[FINAL] Model did not request another tool.

Final Answer:
Lahore is warmer than Islamabad.

Working Memory:
{
  "user_query": "Look up the weather in Lahore and Islamabad and tell me which city is warmer.",
  "iteration": 2,
  "tool_calls": [
    {
      "name": "get_weather",
      "arguments": {
        "city": "Lahore"
      }
    },
    {
      "name": "get_weather",
      "arguments": {
        "city": "Isl

In [108]:
print(
    json.dumps(
        state,
        indent=2,
        default=str
    )
)

{
  "user_query": "Look up the weather in Lahore and Islamabad and tell me which city is warmer.",
  "iteration": 2,
  "tool_calls": [
    {
      "name": "get_weather",
      "arguments": {
        "city": "Lahore"
      }
    },
    {
      "name": "get_weather",
      "arguments": {
        "city": "Islamabad"
      }
    }
  ],
  "observations": [
    {
      "tool": "get_weather",
      "result": {
        "city": "Lahore",
        "temperature_c": 38,
        "condition": "Sunny"
      }
    },
    {
      "tool": "get_weather",
      "result": {
        "city": "Islamabad",
        "temperature_c": 31,
        "condition": "Partly cloudy"
      }
    }
  ],
  "errors": []
}


In [111]:
#maximum iteeration safeguard
result, state = run_agent(
    "Look up the weather in Lahore and Islamabad and tell me which city is warmer.",
    max_iterations=1
)

print("\nReturned Result:")
print(result)

AGENT STARTED
[USER] Look up the weather in Lahore and Islamabad and tell me which city is warmer.
[REASON] Model is deciding what to do. Iteration 1
[ACT] Calling tool 'get_weather' with {'city': 'Lahore'}
[OBSERVE] Tool 'get_weather' returned {'city': 'Lahore', 'temperature_c': 38, 'condition': 'Sunny'}
[ACT] Calling tool 'get_weather' with {'city': 'Islamabad'}
[OBSERVE] Tool 'get_weather' returned {'city': 'Islamabad', 'temperature_c': 31, 'condition': 'Partly cloudy'}
[STATE] Conversation memory updated with tool results.
[ERROR] Maximum iterations reached.

Working Memory:
{
  "user_query": "Look up the weather in Lahore and Islamabad and tell me which city is warmer.",
  "iteration": 1,
  "tool_calls": [
    {
      "name": "get_weather",
      "arguments": {
        "city": "Lahore"
      }
    },
    {
      "name": "get_weather",
      "arguments": {
        "city": "Islamabad"
      }
    }
  ],
  "observations": [
    {
      "tool": "get_weather",
      "result": {
       

## Conversation Memory vs Working Memory

### Conversation Memory

Conversation memory contains the information that is sent back to the model so that it can understand the current task.

In our agent, this is stored in:

`messages`

It contains:

- User message
- Model's function call
- Tool results
- Previous conversation turns

The model uses this information to decide what to do next.

### Working Memory

Working memory is state maintained by our Python application while the agent is running.

In our implementation, it is stored in:

`state`

It contains:

- Current iteration
- Tool calls
- Tool observations
- Errors
- Original user query

Working memory is primarily useful for controlling, monitoring, debugging, and analyzing the agent.

### Key Difference

Conversation memory answers:

"What does the model need to remember?"

Working memory answers:

"What does the application need to track while executing the task?"

# Task 5 — Failure Modes & Guardrails

An agent that only ever sees "happy path" inputs teaches you nothing about how it will
behave in production. This section deliberately breaks `run_agent` three ways —
an ambiguous request, a request that triggers a real tool error, and a request that
needs a tool we never built — and documents what actually happened, not what we
expected to happen.

In [116]:
print("### BREAK TEST 1 — AMBIGUOUS REQUEST ###\n")
result, state = run_agent("What's the weather like?")
print("\nReturned result:", result)
print("Tool calls made:", state["tool_calls"])

### BREAK TEST 1 — AMBIGUOUS REQUEST ###

AGENT STARTED
[USER] What's the weather like?
[REASON] Model is deciding what to do. Iteration 1
[ERROR] Model call failed: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=get_weather{"city": "New York"}</function>'}}

Working Memory:
{
  "user_query": "What's the weather like?",
  "iteration": 1,
  "tool_calls": [],
  "observations": [],
  "errors": [
    {
      "error": "Model call failed (likely malformed tool-call generation).",
      "details": "Error code: 400 - {'error': {'message': \"Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.\", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=get_weather{\"city\": \"New York\"}</function>'}}"
    }
  ]
}

Returned result: I'm s

In [117]:
print("### BREAK TEST 2 — TOOL ERROR ###\n")

result, state = run_agent("What's the weather in Faisalabad?")

print("\nReturned result:", result)
print("Observations:", state["observations"])
print("Errors recorded:", state["errors"])

### BREAK TEST 2 — TOOL ERROR ###

AGENT STARTED
[USER] What's the weather in Faisalabad?
[REASON] Model is deciding what to do. Iteration 1
[ACT] Calling tool 'get_weather' with {'city': 'Faisalabad'}
[OBSERVE] Tool 'get_weather' returned {'error': 'Weather data unavailable for Faisalabad.'}
[STATE] Conversation memory updated with tool results.
[REASON] Model is deciding what to do. Iteration 2
[ACT] Calling tool 'get_weather' with {'city': 'Faisalabad, Pakistan'}
[OBSERVE] Tool 'get_weather' returned {'error': 'Weather data unavailable for Faisalabad, Pakistan.'}
[STATE] Conversation memory updated with tool results.
[REASON] Model is deciding what to do. Iteration 3
[FINAL] Model did not request another tool.

Final Answer:
The weather in Faisalabad, Pakistan is not available at the moment. I recommend checking a weather website or app for the most up-to-date information.

Working Memory:
{
  "user_query": "What's the weather in Faisalabad?",
  "iteration": 3,
  "tool_calls": [
   

In [118]:
print("### BREAK TEST 3 — MISSING TOOL ###\n")

result, state = run_agent("Send an email summary of today's weather in Lahore to my manager.")

print("\nReturned result:", result)
print("Tool calls made:", state["tool_calls"])
print("Errors recorded:", state["errors"])

### BREAK TEST 3 — MISSING TOOL ###

AGENT STARTED
[USER] Send an email summary of today's weather in Lahore to my manager.
[REASON] Model is deciding what to do. Iteration 1
[ACT] Calling tool 'get_weather' with {'city': 'Lahore'}
[OBSERVE] Tool 'get_weather' returned {'city': 'Lahore', 'temperature_c': 38, 'condition': 'Sunny'}
[STATE] Conversation memory updated with tool results.
[REASON] Model is deciding what to do. Iteration 2
[FINAL] Model did not request another tool.

Final Answer:
Here is a summary of today's weather in Lahore:

"Dear Manager, 

Today's weather in Lahore is sunny with a temperature of 38 degrees Celsius.

Best regards,
[Your Name]"

Please note that you should replace [Your Name] with your actual name.

Working Memory:
{
  "user_query": "Send an email summary of today's weather in Lahore to my manager.",
  "iteration": 2,
  "tool_calls": [
    {
      "name": "get_weather",
      "arguments": {
        "city": "Lahore"
      }
    }
  ],
  "observations": [


In [119]:
print("### BREAK TEST 4 — MALFORMED TOOL INPUT ###\n")

print(calculator("25 + "))
print(calculator("import os"))
print(calculator("2 / 0"))

### BREAK TEST 4 — MALFORMED TOOL INPUT ###

{'error': 'invalid syntax (<unknown>, line 1)'}
{'error': 'invalid syntax (<unknown>, line 1)'}
{'error': 'division by zero'}


In [120]:
print("### BREAK TEST 5 — FORCED NON-CONVERGENCE ###\n")

result, state = run_agent(
    "Look up the weather in Lahore and Islamabad and tell me which city is warmer.",
    max_iterations=1
)

print("\nReturned result:", result)
print("Iteration reached:", state["iteration"])

### BREAK TEST 5 — FORCED NON-CONVERGENCE ###

AGENT STARTED
[USER] Look up the weather in Lahore and Islamabad and tell me which city is warmer.
[REASON] Model is deciding what to do. Iteration 1
[ACT] Calling tool 'get_weather' with {'city': 'Lahore'}
[OBSERVE] Tool 'get_weather' returned {'city': 'Lahore', 'temperature_c': 38, 'condition': 'Sunny'}
[ACT] Calling tool 'get_weather' with {'city': 'Islamabad'}
[OBSERVE] Tool 'get_weather' returned {'city': 'Islamabad', 'temperature_c': 31, 'condition': 'Partly cloudy'}
[STATE] Conversation memory updated with tool results.
[ERROR] Maximum iterations reached.

Working Memory:
{
  "user_query": "Look up the weather in Lahore and Islamabad and tell me which city is warmer.",
  "iteration": 1,
  "tool_calls": [
    {
      "name": "get_weather",
      "arguments": {
        "city": "Lahore"
      }
    },
    {
      "name": "get_weather",
      "arguments": {
        "city": "Islamabad"
      }
    }
  ],
  "observations": [
    {
      "

## Task 5 — Failure Modes Observed

| # | Failure Mode | What Actually Happened | Mitigation |
|---|---|---|---|
| 1 | **Malformed function-call generation** | Asked "What's the weather like?" (no city). Model guessed `"New York"` but emitted a broken tool-call format, causing a `400 tool_use_failed` before our code even ran. Caught by try/except, returned a graceful fallback message. | Wrap every model call in try/except (done) — degrade to a fallback message instead of crashing |
| 2 | **Tool error → honest failure (not hallucinated)** | Asked about Faisalabad — tool errored, model retried once with a reformatted city, errored again, then honestly reported the data wasn't available. Did **not** fabricate a temperature. | Good outcome, but the retry wasted an iteration. Prompt the model: on tool error, report immediately, don't retry with a variant |
| 3 | **Missing tool → silent scope creep** | Asked to "send an email" (no email tool exists). Model didn't hallucinate a tool call — it silently got weather instead and wrote an email **as text**, never sending it or flagging the missing capability. `state["errors"]` stayed empty. | Most dangerous finding — no error signal at all. System prompt should list what the agent *can't* do and require it to state that limitation explicitly |
| 4 | **Malformed tool input** | `calculator("25 + ")` → syntax error; `calculator("import os")` → syntax error (blocked at parse stage); `calculator("2 / 0")` → division-by-zero error. All three failed safely, no crash. | AST whitelist blocks both invalid and malicious input before evaluation — confirmed working |
| 5 | **Non-convergence from a hard cutoff** | With `max_iterations=1`, model fetched both cities' weather in one iteration but got cut off before it could compare and answer — despite having all needed data. | Iteration count alone is blunt. Check if required data is already in `observations` before hard-stopping, or allow one final synthesis call |
| 6 | **Silent errors (general)** | Confirmed structurally: every `except` block converts failures into logged, recorded `{"error": ...}` dicts (seen in tests 1 and 4). | Already implemented — but test 3 shows logging alone can't catch a *silent substitution*, only silent exceptions |

## Why Frameworks Exist

**The problems are real**
Everything in Task 5 above — checking tool names against a registry, catching
exceptions, capping iterations, logging each step — is logic we had to write by hand,
and every one of those checks is easy to forget, get subtly wrong, or skip under
deadline pressure on the next project.

**What frameworks actually solve.**
Frameworks like **LangChain**, **LangGraph**, and **CrewAI** exist because these
guardrails are the same handful of problems every agent project runs into — safe tool
dispatch, structured retries, loop detection, memory management, multi-agent
handoffs — and having a battle-tested, reusable implementation of them means you're
not reinventing (and re-debugging) the same error handling every time.

**What they don't do differently.**
They don't do anything conceptually different from `run_agent()` above; they just wrap
the ReAct loop in more scaffolding, more defaults, and more edge-case handling than a
one-day raw-Python version can reasonably include.

**Why building it by hand first still mattered.**
Building the raw version first, as we did today, is what makes that scaffolding
legible instead of magic — you now know exactly which problem each piece of a
framework's abstraction is solving, because you've hit that problem yourself.